# NeoN Python coverage: benchmark `laplacian(gamma, U) - div(phi, U)`

This notebook now tracks every major element in the C++ benchmark snippet:
1. executor sweep and size sweep metadata,
2. `nCells = 10` mesh and `U`, `phi`, `gamma` setup,
3. unoptimized and fused (optimized) expression paths,
4. runtime scheme tokens (`Gauss`, `linear`, `uncorrected`, `upwind`),
5. benchmark-style loops over executors and sizes.

> Current Python bindings expose operator construction but not `Expression.read(...)`, `Expression.assemble(...)`, or `dsl::optimize(...)`. This notebook explicitly demonstrates parity where available and marks the non-exposed steps.

## 1) Imports and runtime init

In [ ]:
import numpy as np
import neon

if not globals().get("_neon_initialized", False):
    neon.initialize()
    _neon_initialized = True

exec = neon.SerialExecutor()
mesh = neon.create_1d_uniform_mesh(exec, 8)

print("executor:", exec.name())
print("cells:", mesh.n_cells())

## 2) Benchmark metadata: sizes, epsilon, and available executors

In [ ]:
epsilon = 1e-32
sizes = [1 << 16, 1 << 17, 1 << 18, 1 << 19, 1 << 20]

executors = [("SerialExecutor", neon.SerialExecutor()), ("CPUExecutor", neon.CPUExecutor())]
if neon.gpu_available():
    executors.append(("GPUExecutor", neon.GPUExecutor()))

print("epsilon:", epsilon)
print("sizes:", sizes)
print("executors:", [name for name, _ in executors])


## 3) Mirror C++ field setup (`nCells = 10`, `nFaces = 9`)

In [ ]:
exec_name, exec = executors[0]
n_cells = 10
mesh = neon.create_1d_uniform_mesh(exec, n_cells)

vol_bcs = neon.create_calculated_volume_bcs_scalar(mesh)
surf_bcs = neon.create_calculated_surface_bcs_scalar(mesh)

U = neon.ScalarVolumeField(exec, "U", mesh)
phi = neon.ScalarSurfaceField(exec, "phi", mesh)
gamma = neon.ScalarSurfaceField(exec, "gamma", mesh)

neon.fill(U.internal_vector(), 2.0)
neon.fill(phi.internal_vector(), 1.0)
neon.fill(gamma.internal_vector(), 2.0)

n_faces = mesh.n_internal_faces() + mesh.n_boundary_faces()
print("executor:", exec_name)
print("nCells:", mesh.n_cells())
print("nFaces (mesh):", n_faces, "(benchmark constant is 9)")
print("volume BC count:", len(vol_bcs), "surface BC count:", len(surf_bcs))


## 4) Unoptimized expression path (`laplacian - div`)

In [ ]:
lap_op = imp.laplacian(gamma, U)
div_op = imp.div(phi, U)
expr_unoptimized = lap_op - div_op

print("laplacian operator:", lap_op.get_name())
print("divergence operator:", div_op.get_name())
print("unoptimized expression terms:", expr_unoptimized.size())


## 5) Fused path mapping (`dsl::optimize(expr)` in C++)

In [ ]:
expr_fused = expr_unoptimized
optimize_exposed = hasattr(neon, "optimize")

print("Python exposes neon.optimize:", optimize_exposed)
if optimize_exposed:
    expr_fused = neon.optimize(expr_unoptimized)
    print("fused expression terms:", expr_fused.size())
else:
    print("using expr_unoptimized as fused placeholder (binding not exposed yet)")


## 6) Scheme-token coverage (`Gauss`, `linear`, `uncorrected`, `upwind`)

In [ ]:
lap_tokens = neon.TokenList()
lap_tokens.insert_string("Gauss")
lap_tokens.insert_string("linear")
lap_tokens.insert_string("uncorrected")

div_tokens = neon.TokenList()
div_tokens.insert_string("Gauss")
div_tokens.insert_string("upwind")

print("laplacian(gamma,U) tokens:", [lap_tokens.get_string(i) for i in range(lap_tokens.size())])
print("div(phi,U) tokens:", [div_tokens.get_string(i) for i in range(div_tokens.size())])

print("read(Dictionary) coverage note: nested Dictionary->TokenList scheme trees are not exposed in Python yet")


## 7) Benchmark-style loop coverage (executors x sizes)

In [ ]:
rows = []
for exec_name, exec in executors:
    mesh = neon.create_1d_uniform_mesh(exec, 10)
    U = neon.ScalarVolumeField(exec, "U", mesh)
    phi = neon.ScalarSurfaceField(exec, "phi", mesh)
    gamma = neon.ScalarSurfaceField(exec, "gamma", mesh)
    neon.fill(U.internal_vector(), 2.0)
    neon.fill(phi.internal_vector(), 1.0)
    neon.fill(gamma.internal_vector(), 2.0)

    expr_unopt = imp.laplacian(gamma, U) - imp.div(phi, U)
    expr_fused = neon.optimize(expr_unopt) if hasattr(neon, "optimize") else expr_unopt

    for size in sizes:
        rows.append((exec_name, size, expr_unopt.size(), expr_fused.size()))

print("rows produced:", len(rows))
print("first 5 rows:")
for row in rows[:5]:
    print(row)

print("assemble coverage note: Expression.assemble(...) is not exposed in Python bindings yet")


## 8) Optional finalize

In [ ]:
if globals().get("_neon_initialized", False):
    neon.finalize()
    _neon_initialized = False
    print("NeoN finalized")
else:
    print("NeoN already finalized")
